# MedGemma QLoRA Fine-tuning for Adaptive NeSy-Gen

This notebook performs task-specific multimodal SFT with 4-bit QLoRA. The medical vision encoder is frozen; decoder LoRA adapters are trained on Findings targets. Retrieval-conditioned examples use only visually retrieved training studies and exclude the same underlying study. Hyperparameters are selected on `val`; `test` is reserved for one final run.

## Cell 1: Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: Configuration

In [ ]:
from pathlib import Path

RUN_DATASET = 'iuxray_official'  # iuxray_official or mimic_aug
SMOKE_RUN = True                 # 256 train / 64 val examples
FINAL_TEST_EVALUATION = False    # set True only after choosing hyperparameters on val

BASE_MODEL = 'google/medgemma-4b-it'
MEDSIGLIP_MODEL = 'google/medsiglip-448'
RETRIEVAL_PROBABILITY = 0.50
RETRIEVAL_TOP_K = 5
FEW_SHOT_K = 3
LORA_RANK = 16
LORA_ALPHA = 16
LEARNING_RATE = 2e-4
EPOCHS = 1 if SMOKE_RUN else 3
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 8
EVAL_STEPS = 10 if SMOKE_RUN else 100
MAX_TRAIN_EXAMPLES = 256 if SMOKE_RUN else None
MAX_EVAL_EXAMPLES = 64 if SMOKE_RUN else None
GENERATION_LIMIT = 25 if SMOKE_RUN else None

REPO_DIR = Path('/content/Nesy-GenRevision')
AAAI_ROOT = Path('/content/drive/MyDrive/aaai_2026_experiments')
SOURCE_RUN = AAAI_ROOT / RUN_DATASET / 'aaai_vision_t5_base_convnext_primekg'
MANIFEST_NAME = 'iuxray_official_manifest.jsonl' if RUN_DATASET == 'iuxray_official' else 'mimic_aug_manifest.jsonl'
MANIFEST = SOURCE_RUN / MANIFEST_NAME
RAD_PRIMEKG_DIR = Path(f'/content/drive/MyDrive/primekg_radiology_cache_{RUN_DATASET}')
RUN_DIR = AAAI_ROOT / RUN_DATASET / 'medgemma_qlora_r16'
TRAIN_DIR = RUN_DIR / 'training'
ADAPTER_DIR = TRAIN_DIR / 'final_adapter'
MEDSIGLIP_CACHE = Path(f'/content/drive/MyDrive/medsiglip_cache_{RUN_DATASET}/train_index.npz')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Manifest:', MANIFEST)
print('Adapter:', ADAPTER_DIR)

## Cell 3: Hugging Face Authentication

In [ ]:
import os, subprocess, sys, time
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
from google.colab import userdata
from huggingface_hub import login
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Add HF_TOKEN in Colab Secrets and grant notebook access.'
login(token=HF_TOKEN)

## Cell 4: Install Repository and Training Dependencies

In [ ]:
import subprocess, sys
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/FaezehMillerAI/Nesy-GenRevision.git', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[finetune,eval,colab]'], check=True)
print('Repository ready:', REPO_DIR)

## Cell 5: Validate GPU and Data Assets

In [ ]:
import torch
required = [MANIFEST, RAD_PRIMEKG_DIR / 'kg.csv', RAD_PRIMEKG_DIR / 'nodes.csv']
for path in required: print(path, '->', path.exists())
assert all(path.exists() for path in required), 'Correct missing paths in Cell 2.'
assert torch.cuda.is_available(), 'Select a GPU runtime.'
memory_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print('GPU:', torch.cuda.get_device_name(0), f'{memory_gb:.1f} GB')
assert torch.cuda.is_bf16_supported(), 'QLoRA requires a bfloat16-capable GPU.'
if memory_gb < 39: print('Warning: use an A100 40 GB runtime or reduce batch/sequence size.')

## Cell 6: Train the QLoRA Adapter

The test split is rejected by the training script. The first run builds and persists the frozen MedSigLIP training index.

In [ ]:
import os, subprocess, sys
TRAIN_LOG = RUN_DIR / 'qlora_training.log'
cmd = [sys.executable, '-u', 'scripts/train_medgemma_lora.py',
    '--manifest', str(MANIFEST), '--output-dir', str(TRAIN_DIR),
    '--model-name', BASE_MODEL, '--train-split', 'train', '--eval-split', 'val',
    '--medsiglip-model', MEDSIGLIP_MODEL, '--retrieval-cache', str(MEDSIGLIP_CACHE),
    '--retrieval-top-k', str(RETRIEVAL_TOP_K), '--few-shot-k', str(FEW_SHOT_K),
    '--retrieval-probability', str(RETRIEVAL_PROBABILITY),
    '--lora-rank', str(LORA_RANK), '--lora-alpha', str(LORA_ALPHA),
    '--learning-rate', str(LEARNING_RATE), '--epochs', str(EPOCHS),
    '--train-batch-size', str(TRAIN_BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRADIENT_ACCUMULATION),
    '--eval-steps', str(EVAL_STEPS)]
if MAX_TRAIN_EXAMPLES: cmd += ['--max-train-examples', str(MAX_TRAIN_EXAMPLES)]
if MAX_EVAL_EXAMPLES: cmd += ['--max-eval-examples', str(MAX_EVAL_EXAMPLES)]
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
started = time.perf_counter()
with TRAIN_LOG.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=0, env=env)
    while True:
        chunk = process.stdout.read(1)
        if chunk == '' and process.poll() is not None: break
        if chunk: print(chunk, end='', flush=True); log_file.write(chunk); log_file.flush()
    return_code = process.wait()
if return_code != 0: raise RuntimeError(f'Training failed. Inspect {TRAIN_LOG}')
assert (ADAPTER_DIR / 'adapter_config.json').exists()
print('Saved adapter:', ADAPTER_DIR)
print(f'Cell 6 elapsed: {(time.perf_counter() - started) / 3600:.2f} hours')

## Cell 7: Generate Validation Reports with the Adapter

In [ ]:
import subprocess, sys
VAL_DIR = RUN_DIR / 'validation'
VAL_DIR.mkdir(parents=True, exist_ok=True)
VAL_PREDICTIONS = VAL_DIR / 'predictions.csv'
cmd = [sys.executable, '-u', 'scripts/generate_medgemma_adaptive_reports.py',
    '--manifest', str(MANIFEST), '--primekg-dir', str(RAD_PRIMEKG_DIR), '--split', 'val',
    '--medgemma-model', BASE_MODEL, '--medgemma-adapter', str(ADAPTER_DIR),
    '--draft-mode', 'few_shot', '--retrieval-cache', str(MEDSIGLIP_CACHE),
    '--output-csv', str(VAL_PREDICTIONS), '--candidates-csv', str(VAL_DIR / 'candidates.csv'),
    '--claim-trace-jsonl', str(VAL_DIR / 'claim_traces.jsonl'),
    '--claim-audit-csv', str(VAL_DIR / 'claim_audit.csv')]
if GENERATION_LIMIT: cmd += ['--limit', str(GENERATION_LIMIT)]
subprocess.run(cmd, cwd=REPO_DIR, check=True)
print('Validation predictions:', VAL_PREDICTIONS)

## Cell 8: Validation Metrics and Leakage Audit

In [ ]:
import json, subprocess, sys
import pandas as pd
VAL_METRICS = VAL_DIR / 'metrics_verified.json'
RAW_PREDICTIONS = VAL_DIR / 'predictions_raw.csv'
raw_frame = pd.read_csv(VAL_PREDICTIONS)
raw_frame['prediction'] = raw_frame['raw_prediction']
raw_frame.to_csv(RAW_PREDICTIONS, index=False)
RAW_METRICS = VAL_DIR / 'metrics_raw.json'
subprocess.run([sys.executable, 'scripts/evaluate_generation.py', '--manifest', str(MANIFEST),
    '--predictions-csv', str(RAW_PREDICTIONS), '--output-json', str(RAW_METRICS),
    '--nodes-csv', str(RAD_PRIMEKG_DIR / 'nodes.csv')], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, 'scripts/evaluate_generation.py', '--manifest', str(MANIFEST),
    '--predictions-csv', str(VAL_PREDICTIONS), '--output-json', str(VAL_METRICS),
    '--nodes-csv', str(RAD_PRIMEKG_DIR / 'nodes.csv')], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, 'scripts/audit_prediction_leakage.py', '--manifest', str(MANIFEST),
    '--predictions-csv', str(VAL_PREDICTIONS), '--output-json', str(VAL_DIR / 'leakage.json'),
    '--high-overlap-threshold', '0.95'], cwd=REPO_DIR, check=True)
raw_metrics = json.loads(RAW_METRICS.read_text())['lexical_metrics']
verified_metrics = json.loads(VAL_METRICS.read_text())['lexical_metrics']
print('Raw fine-tuned draft:', json.dumps(raw_metrics, indent=2))
print('After adaptive verification:', json.dumps(verified_metrics, indent=2))
print('Tune only from these validation metrics; do not inspect test metrics yet.')

## Cell 9: One Final Test Run

Set `FINAL_TEST_EVALUATION=True` in Cell 2 only after fixing the model and hyperparameters.

In [ ]:
assert FINAL_TEST_EVALUATION, 'Keep this False until validation-based model selection is complete.'
TEST_DIR = RUN_DIR / 'final_test'
TEST_DIR.mkdir(parents=True, exist_ok=True)
TEST_PREDICTIONS = TEST_DIR / 'predictions.csv'
cmd = [sys.executable, '-u', 'scripts/generate_medgemma_adaptive_reports.py',
    '--manifest', str(MANIFEST), '--primekg-dir', str(RAD_PRIMEKG_DIR), '--split', 'test',
    '--medgemma-model', BASE_MODEL, '--medgemma-adapter', str(ADAPTER_DIR),
    '--draft-mode', 'few_shot', '--retrieval-cache', str(MEDSIGLIP_CACHE),
    '--output-csv', str(TEST_PREDICTIONS), '--candidates-csv', str(TEST_DIR / 'candidates.csv'),
    '--claim-trace-jsonl', str(TEST_DIR / 'claim_traces.jsonl'),
    '--claim-audit-csv', str(TEST_DIR / 'claim_audit.csv')]
subprocess.run(cmd, cwd=REPO_DIR, check=True)
TEST_METRICS = TEST_DIR / 'metrics.json'
subprocess.run([sys.executable, 'scripts/evaluate_generation.py', '--manifest', str(MANIFEST),
    '--predictions-csv', str(TEST_PREDICTIONS), '--output-json', str(TEST_METRICS),
    '--nodes-csv', str(RAD_PRIMEKG_DIR / 'nodes.csv'), '--prepare-official-inputs-dir', str(TEST_DIR / 'official_inputs')], cwd=REPO_DIR, check=True)
print(json.dumps(json.loads(TEST_METRICS.read_text())['lexical_metrics'], indent=2))